In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/databricks-novacart/02.Configurations

In [0]:
bronze_orders = f"{catalog}.{bronze_schema}.orders"
silver_orders = f"{catalog}.{silver_schema}.orders"
bronze_payments = f"{catalog}.{bronze_schema}.payments"
silver_payments = f"{catalog}.{silver_schema}.payments"

In [0]:
from pyspark.sql.functions import col,lit,current_timestamp,lower,max
from delta.tables import DeltaTable
import uuid
from datetime import datetime,UTC

In [0]:
silver_run_id = str(uuid.uuid4())

In [0]:
spark.sql("""
          create table if not exists novacart.silver.ingestion_control(
              table_name string,
              last_processed_at timestamp,
              silver_run_id string,
              status string,
              updated_at timestamp
          )
          """)

In [0]:
def get_last_processed_at(table_name):
        cntrl = (
            spark.table("novacart.silver.ingestion_control")
                .filter((col("status") == "success") & (col("table_name") == lit(table_name)))
                .limit(1)
        )
        rows = cntrl.collect()
        if not rows:
            return None
        
        return rows[0]["last_processed_at"]


In [0]:
def upsert_to_silver(source_df,target_table,join_condiiton):
    if spark.catalog.tableExists(target_table):
       delta_table = DeltaTable.forName(spark,target_table)
       (
           delta_table.alias("t")
                      .merge(source_df.alias("s"),join_condiiton)
                      .whenMatchedUpdateAll()
                      .whenNotMatchedInsertAll()
                      .execute()
       )
    else:
        source_df.write.format("delta").saveAsTable(target_table)

In [0]:
def upsert_to_silver_control(table_name,last_processed_at,silver_run_id):
    src_df = (spark.createDataFrame([(
        table_name,
        last_processed_at,
        silver_run_id,
        "success",
        datetime.now(UTC)
    )],schema='table_name string,last_processed_at timestamp,silver_run_id string,status string,updated_at timestamp'
                          )
    )
    delta_table = DeltaTable.forName(spark,"novacart.silver.ingestion_control")
    (
        delta_table.alias("t")
                   .merge(src_df.alias("s"),"t.table_name = s.table_name")
                   .whenMatchedUpdate(
                       set = {"t.last_processed_at":"s.last_processed_at",
                              "t.silver_run_id":"s.silver_run_id",
                              "t.updated_at":"s.updated_at",
                              "t.status":"s.status"}
                   )
                   .whenNotMatchedInsertAll()
                   .execute()
    )


In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_processed_at = get_last_processed_at(entity_name)
    bronze_df = spark.table(bronze_table)
    if last_processed_at is None:
        return bronze_df,last_processed_at
    else:
        return bronze_df.filter(col("bronze_ingested_at") > lit(last_processed_at)),last_processed_at

#### Load Orders data

In [0]:
orders_inc,last_processed_at = get_incremental_bronze(bronze_orders,"orders")
if orders_inc.count() > 0:
    orders_cleaned = orders_inc.withColumn("order_status",lower(col("order_status")))
    upsert_to_silver(orders_cleaned,silver_orders,"t.order_id = s.order_id")
    mx_ingested = orders_cleaned.agg(max("bronze_ingested_at").alias("mx_date")).orderBy(col("mx_date").desc()).collect()[0]["mx_date"]
    upsert_to_silver_control("orders",mx_ingested,silver_run_id)
else:
    print("No new records found")
    upsert_to_silver_control("orders",last_processed_at,silver_run_id)

#### Load Payments data

In [0]:
payments_inc,last_processed_at = get_incremental_bronze(bronze_payments,"payments")
if payments_inc.count() > 0:
    payments_cleaned = payments_inc.withColumn("payment_status",lower(col("payment_status")))
    upsert_to_silver(payments_cleaned,silver_payments,"t.payment_id = s.payment_id")
    mx_ingested = payments_cleaned.agg(max("bronze_ingested_at").alias("mx_date")).orderBy(col("mx_date").desc()).collect()[0]["mx_date"]
    print("No of rows loaded for paymnet",payments_inc.count())
    upsert_to_silver_control("payments",mx_ingested,silver_run_id)
else:
    print("No new records found")
    upsert_to_silver_control("payments",last_processed_at,silver_run_id)

In [0]:
%sql
select * from novacart.silver.payments